# 📝 Notebook 4: JSONL Dataset Creation
**Goal:** Convert cleaned data into QA/Translation/Summarization format with mandatory safety sentences.

**Output:** `data/train.jsonl`, `data/val.jsonl`, `data/test.jsonl`

In [ ]:
import os, json, re, pickle, random
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset

random.seed(42)
os.makedirs('data', exist_ok=True)
print('✅ Libraries loaded')

## 1. Load Resources

In [ ]:
# Load clean dataset
df_clean = pd.read_csv('data/clean_gu_health.csv', encoding='utf-8')
df_anuvaad = df_clean[df_clean['source'] == 'anuvaad'].copy()
df_indicnlp = df_clean[df_clean['source'] == 'indicnlp'].copy()

# Load KG
with open('data/kg.pkl', 'rb') as f:
    G = pickle.load(f)

print(f'Clean dataset: {len(df_clean):,} sentences')
print(f'  Anuvaad (parallel): {len(df_anuvaad):,}')
print(f'  IndicNLP (mono):    {len(df_indicnlp):,}')
print(f'Knowledge Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

## 2. Safety Sentence Templates

In [ ]:
# Gujarati safety sentences to append to all answers
SAFETY_SENTENCES = [
    'કૃપા કરીને યોગ્ય ઉપચાર માટે ડૉક્ટરની સલાહ લો.',                              # Please consult a doctor for proper treatment.
    'આ માહિતી ફક્ત શૈક્ષણિક હેતુ માટે છે; ચિકિત્સક સાથે પ્રત્યક્ષ સંપર્ક કરો.',  # Educational only; contact physician directly.
    'ગંભીર લક્ષણો હોય તો તાત્કાલિક હૉસ્પિટલ જાઓ.',                               # For severe symptoms, go to hospital immediately.
    'ઘરેલૂ ઉપાય પ્રથમ સહાય માટે છે, ડૉક્ટરની સલાહ ફરજિયાત છે.',                  # Home remedies are first-aid; doctor consult mandatory.
    'કોઈ પણ દવા ડૉક્ટરની પ્રિસ્ક્રિપ્શન વિના ન લો.',                              # Do not take any medicine without prescription.
    'તમારા નજીકના ઉત્ સ્વાસ્થ્ય કેન્દ્ર અથવા ડૉક્ટરનો સત્વરે સંપર્ક કરો.',      # Contact nearest health center or doctor promptly.
]

def get_safety_sentence() -> str:
    return random.choice(SAFETY_SENTENCES)

print(f'Safety sentence pool: {len(SAFETY_SENTENCES)} templates')

## 3. Task A — QA Examples (from IndicQA + KG)

In [ ]:
qa_examples = []

# 3a. Load IndicQA Gujarati (if available)
try:
    indicqa = load_dataset('ai4bharat/IndicQA', 'gu', trust_remote_code=True)
    qa_data = indicqa.get('test', indicqa.get('validation', None))
    print(f'IndicQA loaded: {len(qa_data)} examples')
    for item in tqdm(qa_data, desc='IndicQA'):
        question = item.get('question', '').strip()
        answer = item.get('answers', {}).get('text', [''])[0].strip()
        context = item.get('context', '').strip()
        if question and answer and len(answer) > 5:
            qa_examples.append({
                'task': 'qa',
                'instruction': f'નીચેના સંદર્ભ ના આધારે પ્રશ્નનો ઉત્તર ગુજરાતીમાં આપો:\nSandarabh: {context[:300]}\nPrashn: {question}',
                'output': f'{answer} {get_safety_sentence()}'
            })
except Exception as e:
    print(f'IndicQA load failed: {e}. Using KG-based QA only.')

In [ ]:
# 3b. Generate QA from KG (symptom → disease → treatment)
import networkx as nx

QA_TEMPLATES = [
    # (question_template, answer_template)
    ("{symptom} ક્યા રોગ ના લક્ષણ હોઈ શકે?",
     "{symptom} {disease_list} ના સંભવિત સૂચક છે."),
    ("{disease} ની સારવાર ક્યા છે?",
     "{disease} ની સારવારમાં {treatment_list} નો ઉપયોગ થઈ શકે છે."),
    ("{disease} ના મુખ્ય લક્ષણો ક્યા છે?",
     "{disease} ના સામાન્ય લક્ષણોમાં {symptom_list} સામેલ છે."),
    ("{symptom} હોય ત્યારે ડૉક્ટર ક્યો ઉપચાર સૂચવી શકે?",
     "{symptom} ના કારણ {disease_list} હોઈ શકે; સારવારમાં {treatment_list} સામેલ હોઈ શકે."),
]

# Find symptom nodes
symptom_nodes = [n for n, d in G.nodes(data=True) if d.get('type') == 'symptom']
disease_nodes = [n for n, d in G.nodes(data=True) if d.get('type') == 'disease']

for symptom in tqdm(symptom_nodes, desc='KG QA (symptom-based)'):
    diseases = [nbr for nbr in G.successors(symptom) if G.nodes[nbr].get('type') == 'disease']
    if not diseases: continue
    disease = random.choice(diseases)
    treatments = [nbr for nbr in G.successors(disease) if G.nodes[nbr].get('type') == 'treatment']

    disease_list = ', '.join(diseases[:3]) or disease
    treatment_list = ', '.join(treatments[:3]) if treatments else 'ડૉક્ટરની સૂચના'

    # Q1: symptom → possible disease?
    qa_examples.append({
        'task': 'qa',
        'instruction': QA_TEMPLATES[0][0].format(symptom=symptom),
        'output': QA_TEMPLATES[0][1].format(symptom=symptom, disease_list=disease_list)
                  + ' ' + get_safety_sentence()
    })

    # Q2: disease → treatment?
    if treatments:
        qa_examples.append({
            'task': 'qa',
            'instruction': QA_TEMPLATES[1][0].format(disease=disease),
            'output': QA_TEMPLATES[1][1].format(disease=disease, treatment_list=treatment_list)
                      + ' ' + get_safety_sentence()
        })

for disease in tqdm(disease_nodes, desc='KG QA (disease-based)'):
    symptoms = [nbr for nbr in G.predecessors(disease) if G.nodes[nbr].get('type') == 'symptom']
    treatments = [nbr for nbr in G.successors(disease) if G.nodes[nbr].get('type') == 'treatment']
    if symptoms:
        symptom_list = ', '.join(symptoms[:3])
        qa_examples.append({
            'task': 'qa',
            'instruction': QA_TEMPLATES[2][0].format(disease=disease),
            'output': QA_TEMPLATES[2][1].format(disease=disease, symptom_list=symptom_list)
                      + ' ' + get_safety_sentence()
        })

print(f'✅ Total QA examples: {len(qa_examples):,}')

## 4. Task B — Translation Examples (en → gu)

In [ ]:
translation_examples = []

# Use Anuvaad parallel pairs where en is available
translation_pairs = df_anuvaad[df_anuvaad['en'].str.len() > 10].sample(
    n=min(5000, len(df_anuvaad)), random_state=42
)

for _, row in tqdm(translation_pairs.iterrows(), total=len(translation_pairs), desc='Translation'):
    en = str(row['en']).strip()
    gu = str(row['gu']).strip()
    if len(en) < 10 or len(gu) < 10: continue
    translation_examples.append({
        'task': 'translation',
        'instruction': f'નીચેના અંગ્રેજી વાક્ય ને ગુજરાતી માં અનુવાદ કરો:\n{en}',
        'output': gu
    })

print(f'✅ Translation examples: {len(translation_examples):,}')

## 5. Task C — Summarization Examples

In [ ]:
summarization_examples = []

# Group sentences into paragraphs of 3–5 sentences and ask to summarize
all_gu_sentences = df_clean['gu'].tolist()
random.shuffle(all_gu_sentences)

SUMM_TEMPLATES = [
    'નીચેનો ફકરો ગુજરાતીમાં સંક્ષિપ્ત (2-3 વાક્ય) માં સારાંશ આપો:',
    'આ ગુજરાતી ફકરા ના મુખ્ય મુદ્દા ટૂંકમાં જણાવો:',
    'નીચેની વૈદ્યકીય માહિતી ટૂંકાણમાં સમજાવો:',
]

i = 0
max_examples = 2000
while i < len(all_gu_sentences) - 5 and len(summarization_examples) < max_examples:
    chunk_size = random.randint(3, 5)
    chunk = all_gu_sentences[i:i+chunk_size]
    i += chunk_size

    paragraph = ' '.join(chunk)
    if len(paragraph) < 80:
        continue

    # Summary = first sentence + key info (heuristic)
    summary_sentences = chunk[:2]
    summary = ' '.join(summary_sentences)
    if len(summary) < 20:
        continue

    summarization_examples.append({
        'task': 'summarization',
        'instruction': f'{random.choice(SUMM_TEMPLATES)}\n{paragraph}',
        'output': f'{summary} {get_safety_sentence()}'
    })

print(f'✅ Summarization examples: {len(summarization_examples):,}')

## 6. Format as Chat (Qwen Chat Template)

In [ ]:
SYSTEM_PROMPT = (
    'You are a safe, accurate Gujarati healthcare assistant. '
    'Answer ONLY based on provided context. Do NOT hallucinate. '
    'Always include a safety advisory sentence in your response.'
)

def format_as_chat(example: dict) -> dict:
    """Format as Qwen chat JSONL for fine-tuning (TRL SFTTrainer format)."""
    return {
        'messages': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': example['instruction']},
            {'role': 'assistant', 'content': example['output']}
        ],
        'task': example['task']
    }

# Combine all examples
all_examples = qa_examples + translation_examples + summarization_examples
random.shuffle(all_examples)

# Format
all_formatted = [format_as_chat(ex) for ex in all_examples]

print(f'Total examples: {len(all_formatted):,}')
print(f'  QA:            {len(qa_examples):,}')
print(f'  Translation:   {len(translation_examples):,}')
print(f'  Summarization: {len(summarization_examples):,}')

## 7. Train / Val / Test Split & Save

In [ ]:
n = len(all_formatted)
n_train = int(0.85 * n)
n_val = int(0.10 * n)
# rest is test

train_data = all_formatted[:n_train]
val_data   = all_formatted[n_train:n_train + n_val]
test_data  = all_formatted[n_train + n_val:]

def save_jsonl(data: list, path: str):
    with open(path, 'w', encoding='utf-8') as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')

save_jsonl(train_data, 'data/train.jsonl')
save_jsonl(val_data,   'data/val.jsonl')
save_jsonl(test_data,  'data/test.jsonl')

print('\n📊 DATASET SPLIT')
print('='*40)
print(f'Train: {len(train_data):>6,} examples → data/train.jsonl')
print(f'Val:   {len(val_data):>6,} examples → data/val.jsonl')
print(f'Test:  {len(test_data):>6,} examples → data/test.jsonl')
print()

# Validate
with open('data/train.jsonl', 'r', encoding='utf-8') as f:
    sample = json.loads(f.readline())
print('Sample training example:')
print(json.dumps(sample, ensure_ascii=False, indent=2))
print('\n✅ Phase 1 Complete! Run Notebook 05 on Kaggle GPU for QLoRA fine-tuning.')